# Machine Learning for Charity Donor Classification
## Project: Predicting High-Value Donors for *CharityML*

### Overview
In this project, I apply various supervised learning algorithms to accurately model individuals' income using data collected from the 1994 U.S. Census. The primary objective is to construct a predictive model that identifies individuals making more than $50,000 annually, as these individuals are most likely to donate to a non-profit organization. This data-driven approach allows *CharityML* to optimize its outreach efforts, focusing resources on high-propensity donors to maximize fundraising efficiency.

### Technical Highlights
- **Exploratory Data Analysis (EDA):** Uncovering patterns and correlations in demographic and socio-economic data.
- **Data Preprocessing:** Handling skewed distributions, feature scaling, and categorical encoding.
- **Model Selection & Evaluation:** Comparing multiple supervised learners (e.g., AdaBoost, Gradient Boosting, SVM) using accuracy and F-beta scores.
- **Model Optimization:** Fine-tuning hyperparameters using Grid Search to achieve optimal performance.
- **Feature Importance:** Identifying the key drivers of high income and donation propensity.

## 1. Data Exploration
We begin by loading the census dataset and performing an initial investigation to understand the distribution of income levels and identify key features.

In [ ]:
import numpy as np
import pandas as pd
from time import time
from IPython.display import display

# Load the Census dataset
data = pd.read_csv("census.csv")

# Display the first record
display(data.head(n=1))

# Basic Statistics
n_records = data.shape[0]
n_greater_50k = data[data['income'] == '>50K'].shape[0]
n_at_most_50k = data[data['income'] == '<=50K'].shape[0]
greater_percent = (n_greater_50k / n_records) * 100

print(f"Total number of records: {n_records}")
print(f"Individuals making more than $50,000: {n_greater_50k}")
print(f"Individuals making at most $50,000: {n_at_most_50k}")
print(f"Percentage of individuals making more than $50,000: {greater_percent:.2f}%")

## 2. Data Preprocessing
To prepare the data for machine learning, we address skewed distributions in numerical features, scale features to a common range, and encode categorical variables.

In [ ]:
# Log-transform the skewed features
skewed = ['capital-gain', 'capital-loss']
features_log_transformed = pd.DataFrame(data = data)
features_log_transformed[skewed] = data[skewed].apply(lambda x: np.log(x + 1))

# Normalize numerical features
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
numerical = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
features_log_minmax_transform = pd.DataFrame(data = features_log_transformed)
features_log_minmax_transform[numerical] = scaler.fit_transform(features_log_transformed[numerical])

# One-hot encode categorical features
features_final = pd.get_dummies(features_log_minmax_transform.drop('income', axis=1))
income = data['income'].apply(lambda x: 1 if x == '>50K' else 0)

print(f"{len(features_final.columns)} total features after one-hot encoding.")

## 3. Model Performance Evaluation
We evaluate several supervised learning models to identify the most promising candidates for predicting high-income individuals.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import fbeta_score, accuracy_score

# Split the data
X_train, X_test, y_train, y_test = train_test_split(features_final, income, test_size = 0.2, random_state = 0)

def train_predict(learner, sample_size, X_train, y_train, X_test, y_test):
    results = {}
    start = time()
    learner = learner.fit(X_train[:sample_size], y_train[:sample_size])
    end = time()
    results['train_time'] = end - start
    
    start = time()
    predictions_test = learner.predict(X_test)
    end = time()
    results['pred_time'] = end - start
    
    results['acc_test'] = accuracy_score(y_test, predictions_test)
    results['f_test'] = fbeta_score(y_test, predictions_test, beta=0.5)
    
    return results

from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

clf_A = AdaBoostClassifier(random_state=42)
clf_B = GradientBoostingClassifier(random_state=42)
clf_C = SVC(random_state=42)

samples_100 = len(y_train)
for clf in [clf_A, clf_B, clf_C]:
    results = train_predict(clf, samples_100, X_train, y_train, X_test, y_test)
    print(f"{clf.__class__.__name__} results: {results}")

## 4. Model Optimization & Feature Importance
We optimize the best candidate model using Grid Search and identify the most important features contributing to the model's predictions.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer

# Initialize the classifier
clf = GradientBoostingClassifier(random_state=42)

# Hyperparameter tuning
parameters = {'n_estimators': [100, 300, 500], 'learning_rate': [0.1, 0.05, 0.01]}
scorer = make_scorer(fbeta_score, beta=0.5)
grid_obj = GridSearchCV(clf, parameters, scoring=scorer)
grid_fit = grid_obj.fit(X_train, y_train)

best_clf = grid_fit.best_estimator_
best_predictions = best_clf.predict(X_test)

print(f"Optimized Model Accuracy: {accuracy_score(y_test, best_predictions):.4f}")
print(f"Optimized Model F-score: {fbeta_score(y_test, best_predictions, beta=0.5):.4f}")

# Feature Importance
importances = best_clf.feature_importances_
indices = np.argsort(importances)[::-1]
columns = X_train.columns.values[indices[:5]]
values = importances[indices[:5]]

print("Top 5 Most Important Features:")
for i in range(5):
    print(f"{i+1}. {columns[i]} ({values[i]:.4f})")

## 5. Conclusion
The optimized Gradient Boosting model provides a robust solution for identifying potential high-value donors. By focusing on key features like capital gains, education, and age, *CharityML* can significantly improve the efficiency of its fundraising campaigns, ensuring that outreach efforts are targeted towards individuals with the highest propensity to donate.